# Skala AO Screening Rotation Comparison

This notebook loads JSON written by `benchmarks/run_pyscf_ao_screening_rotation_benchmark.py`. It does not construct molecules, load Skala, or execute benchmark workloads.

Generate a full result file before running the analysis:

```bash
/home/jenswehner/micromamba/envs/skala_gpu_python/bin/python \
    benchmarks/run_pyscf_ao_screening_rotation_benchmark.py \
    --label screening
```

The default run records runtime and incremental peak memory for 72 orientations in each of `gpu`, `cpu_dense`, and `cpu_screened`. Add `--smoke` for one orientation per mode or `--preflight-only` to validate geometry and dependencies without measurements.

## 1. Import Analysis Libraries and Configure Paths

Select one or more rotation result files. Runtime is reported in seconds and incremental peak memory in GiB; CPU dense is the numerical and ratio reference.

In [ ]:
from __future__ import annotations

import json
import statistics
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

MODES = ("gpu", "cpu_dense", "cpu_screened")
MEASUREMENTS = ("runtime", "memory")
REFERENCE_MODE = "cpu_dense"
EXPECTED_AOS = 879
EXPECTED_ROUTES = {
    "gpu": "global_ao_screening",
    "cpu_dense": "dense",
    "cpu_screened": "global_ao_screening",
}
TERMINAL_STATUSES = {
    "ok",
    "timeout",
    "oom",
    "error",
    "skipped_after_resource_failure",
}
FINGERPRINT_LABELS = {
    "electron_integral": "Electron integral",
    "xc_energy": "XC energy",
    "vxc_sum": "Vxc sum",
    "vxc_trace": "Vxc trace",
    "vxc_frobenius_norm": "Vxc Frobenius norm",
    "vxc_max_abs": "Vxc max abs",
}
ERROR_TOLERANCES = {
    "cpu_screened": (5e-8, 1e-8),
    "gpu": (2e-7, 1e-7),
}
MODE_COLORS = {
    "gpu": "#C44536",
    "cpu_dense": "#83C5BE",
    "cpu_screened": "#006D77",
}


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "benchmarks"
        ).is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find the Skala repository above {start}")


REPOSITORY_ROOT = find_repository_root(Path.cwd())
RESULTS_DIR = REPOSITORY_ROOT / "benchmarks" / "results"
ARTIFACT_DIR = RESULTS_DIR / "rotation_comparison"
FIGURE_DIR = ARTIFACT_DIR / "figures"
TABLE_DIR = ARTIFACT_DIR / "tables"
COMPARISON_JSON = ARTIFACT_DIR / "comparison.json"

# Replace this list with explicit paths to compare a subset of result files.
SELECTED_RESULT_FILES = sorted(
    RESULTS_DIR.glob("skala-pyscf-ao-screening-rotations-*.json")
)

sns.set_theme(style="whitegrid", context="notebook")
print(f"Selected {len(SELECTED_RESULT_FILES)} rotation result file(s)")
for result_file in SELECTED_RESULT_FILES:
    print(f"  {result_file.name}")

## 2. Load and Validate Benchmark JSON Files

Each file is checked for the rotation schema, provenance, configuration, environment, timestamps, and runner hashes. Malformed and partially completed files remain visible in the validation table.

## 3. Normalize Molecule and Execution-Mode Results

The molecule is fixed at C7H16, so normalization produces one row per result file, orientation, and execution mode. Rows include geometry, AO and grid sizes, routes, statuses, measurements, allocator baselines, and both runtime and memory fingerprints.

## 4. Validate Benchmark Completeness and Status

A full run requires 72 orientations, three modes, and successful runtime and memory records. Smoke and partial runs are accepted but explicitly reported.

## 5. Verify AO Counts, Routes, and Screening Thresholds

Observed AO counts must remain 879. Dense CPU must report `dense`; screened CPU and GPU must report `global_ao_screening`, which is expected because 879 exceeds PySCF's switch size of 800.

In [ ]:
NORMALIZED_COLUMNS = [
    "run_label",
    "commit",
    "branch",
    "dirty",
    "orientation_key",
    "orientation_index",
    "azimuth_degrees",
    "polar_degrees",
    "formula",
    "carbon_count",
    "expected_aos",
    "actual_aos",
    "electron_count",
    "grid_points",
    "mode",
    "selected_route",
    "route_request",
    "switch_size",
    "implementation_sha256",
    "runtime_status",
    "memory_status",
    "runtime_samples_seconds",
    "runtime_seconds",
    "incremental_peak_bytes",
    "incremental_peak_gib",
    "allocator_baseline_bytes",
    "allocator_baseline_gib",
] + [
    f"{measurement}_{fingerprint}"
    for measurement in MEASUREMENTS
    for fingerprint in FINGERPRINT_LABELS
]


def validate_document(path: Path, document: dict[str, Any]) -> list[str]:
    failures: list[str] = []
    required_fields = {
        "configuration",
        "created_at",
        "environment",
        "orientations",
        "runner_hashes",
        "schema_version",
        "source",
        "updated_at",
    }
    missing_fields = sorted(required_fields - document.keys())
    if missing_fields:
        failures.append(f"missing top-level fields: {missing_fields}")
    if document.get("schema_version") != 1:
        failures.append(f"unsupported schema version {document.get('schema_version')}")
    if document.get("benchmark") != "pyscf_ao_screening_rotations":
        failures.append(f"unexpected benchmark marker {document.get('benchmark')!r}")

    configuration = document.get("configuration", {})
    configured_modes = tuple(configuration.get("modes", ()))
    configured_measurements = tuple(configuration.get("measurements", ()))
    if configured_modes and configured_modes != MODES:
        failures.append(f"configured modes are {configured_modes}, expected {MODES}")
    if configured_measurements and configured_measurements != MEASUREMENTS:
        failures.append(
            f"configured measurements are {configured_measurements}, expected {MEASUREMENTS}"
        )

    orientations = document.get("orientations", {})
    expected_count = int(configuration.get("orientation_count", len(orientations)))
    if len(orientations) != expected_count:
        failures.append(
            f"contains {len(orientations)} orientations, configuration requests {expected_count}"
        )
    if not configuration.get("smoke_run", False) and len(orientations) != 72:
        failures.append(
            f"full run contains {len(orientations)} orientations, expected 72"
        )
    coordinate_hashes = [
        orientation.get("coordinate_sha256") for orientation in orientations.values()
    ]
    if len(set(coordinate_hashes)) != len(coordinate_hashes):
        failures.append("orientation coordinate hashes are not unique")

    for orientation_key, orientation in orientations.items():
        observed = orientation.get("observed") or {}
        actual_aos = observed.get("actual_aos")
        if actual_aos is not None and int(actual_aos) != EXPECTED_AOS:
            failures.append(
                f"{orientation_key}: observed {actual_aos} AOs, expected {EXPECTED_AOS}"
            )
        modes = orientation.get("modes", {})
        missing_modes = sorted(set(MODES) - modes.keys())
        if missing_modes:
            failures.append(f"{orientation_key}: missing modes {missing_modes}")
        for mode in MODES:
            mode_record = modes.get(mode, {})
            route = mode_record.get("route", {})
            selected_route = route.get("selected_route")
            if selected_route is not None and selected_route != EXPECTED_ROUTES[mode]:
                failures.append(
                    f"{orientation_key} {mode}: selected {selected_route}, "
                    f"expected {EXPECTED_ROUTES[mode]}"
                )
            switch_size = route.get("pyscf_switch_size")
            if switch_size is not None and EXPECTED_AOS <= int(switch_size):
                failures.append(
                    f"{orientation_key} {mode}: {EXPECTED_AOS} AOs do not exceed "
                    f"reported switch size {switch_size}"
                )
            for measurement in MEASUREMENTS:
                result = mode_record.get(measurement)
                if result is None:
                    failures.append(f"{orientation_key} {mode}: missing {measurement}")
                    continue
                status = result.get("status")
                if status not in TERMINAL_STATUSES:
                    failures.append(
                        f"{orientation_key} {mode} {measurement}: unknown status {status!r}"
                    )
                elif status != "ok":
                    failures.append(
                        f"{orientation_key} {mode} {measurement}: status {status}"
                    )
                if (
                    measurement == "runtime"
                    and status == "ok"
                    and not result.get("runtime_samples_seconds")
                ):
                    failures.append(
                        f"{orientation_key} {mode}: successful runtime has no samples"
                    )
    return failures


def normalize_document(path: Path, document: dict[str, Any]) -> list[dict[str, Any]]:
    base_molecule = document.get("geometry", {}).get("base_molecule", {})
    source = document.get("source", {})
    run_label = str(document.get("run_label") or path.stem)
    rows: list[dict[str, Any]] = []
    for orientation_key, orientation in document.get("orientations", {}).items():
        observed = orientation.get("observed") or {}
        for mode in MODES:
            mode_record = orientation.get("modes", {}).get(mode, {})
            runtime = mode_record.get("runtime", {})
            memory = mode_record.get("memory", {})
            route = mode_record.get("route", {})
            runtime_samples = [
                float(value) for value in runtime.get("runtime_samples_seconds", [])
            ]
            row: dict[str, Any] = {
                "run_label": run_label,
                "commit": source.get("commit"),
                "branch": source.get("branch"),
                "dirty": source.get("dirty"),
                "orientation_key": orientation_key,
                "orientation_index": orientation.get("index"),
                "azimuth_degrees": orientation.get("azimuth_degrees"),
                "polar_degrees": orientation.get("polar_degrees"),
                "formula": base_molecule.get("formula", observed.get("formula")),
                "carbon_count": base_molecule.get(
                    "carbon_count", observed.get("carbon_count")
                ),
                "expected_aos": base_molecule.get("expected_aos", EXPECTED_AOS),
                "actual_aos": observed.get("actual_aos"),
                "electron_count": observed.get("electron_count"),
                "grid_points": observed.get("grid_points"),
                "mode": mode,
                "selected_route": route.get("selected_route"),
                "route_request": route.get("request"),
                "switch_size": route.get("pyscf_switch_size"),
                "implementation_sha256": route.get("implementation_sha256"),
                "runtime_status": runtime.get("status", "missing"),
                "memory_status": memory.get("status", "missing"),
                "runtime_samples_seconds": runtime_samples,
                "runtime_seconds": (
                    statistics.median(runtime_samples) if runtime_samples else np.nan
                ),
                "incremental_peak_bytes": memory.get("incremental_peak_bytes"),
                "incremental_peak_gib": (
                    float(memory["incremental_peak_bytes"]) / 1024**3
                    if memory.get("incremental_peak_bytes") is not None
                    else np.nan
                ),
                "allocator_baseline_bytes": memory.get("allocator_baseline_bytes"),
                "allocator_baseline_gib": (
                    float(memory["allocator_baseline_bytes"]) / 1024**3
                    if memory.get("allocator_baseline_bytes") is not None
                    else np.nan
                ),
            }
            for measurement, result in (("runtime", runtime), ("memory", memory)):
                fingerprint = result.get("fingerprint", {})
                for fingerprint_key in FINGERPRINT_LABELS:
                    row[f"{measurement}_{fingerprint_key}"] = fingerprint.get(
                        fingerprint_key, np.nan
                    )
            rows.append(row)
    return rows


DOCUMENTS: list[tuple[Path, dict[str, Any]]] = []
validation_rows: list[dict[str, str]] = []
normalized_rows: list[dict[str, Any]] = []
for result_file in SELECTED_RESULT_FILES:
    try:
        document = json.loads(result_file.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        validation_rows.append(
            {"run_label": result_file.stem, "failure": f"could not load: {error}"}
        )
        continue
    DOCUMENTS.append((result_file, document))
    normalized_rows.extend(normalize_document(result_file, document))
    failures = validate_document(result_file, document)
    run_label = str(document.get("run_label") or result_file.stem)
    validation_rows.extend(
        {"run_label": run_label, "failure": failure} for failure in failures
    )

run_labels = [
    str(document.get("run_label") or path.stem) for path, document in DOCUMENTS
]
duplicate_run_labels = sorted(
    label for label, count in Counter(run_labels).items() if count > 1
)
if duplicate_run_labels:
    raise ValueError(f"Run labels must be unique: {duplicate_run_labels}")

normalized_df = pd.DataFrame(normalized_rows, columns=NORMALIZED_COLUMNS)
validation_df = pd.DataFrame(validation_rows, columns=["run_label", "failure"])
if DOCUMENTS:
    print(
        f"Loaded {len(DOCUMENTS)} document(s) and {len(normalized_df)} normalized rows"
    )
else:
    print("No rotation benchmark JSON files found. Run the benchmark first.")
display(
    validation_df
    if not validation_df.empty
    else pd.DataFrame({"validation": ["passed"]})
)

In [ ]:
if normalized_df.empty:
    status_summary_df = pd.DataFrame()
    route_summary_df = pd.DataFrame()
else:
    status_rows: list[dict[str, Any]] = []
    for status_column in ("runtime_status", "memory_status"):
        measurement = status_column.removesuffix("_status")
        grouped = normalized_df.groupby(
            ["run_label", "mode", status_column], dropna=False
        ).size()
        for (run_label, mode, status), count in grouped.items():
            status_rows.append(
                {
                    "run_label": run_label,
                    "mode": mode,
                    "measurement": measurement,
                    "status": status,
                    "count": int(count),
                }
            )
    status_summary_df = pd.DataFrame(status_rows)
    route_summary_df = (
        normalized_df.groupby(["run_label", "mode", "selected_route"], dropna=False)
        .size()
        .rename("orientation_count")
        .reset_index()
    )

print("Measurement status counts")
display(status_summary_df)
print("Selected route counts")
display(route_summary_df)

## 6. Compare Numerical Fingerprints

For every orientation, the runtime and memory workers are compared within each mode. The runtime fingerprints for GPU and screened CPU are also compared with CPU dense at the same orientation. The table reports signed, absolute, and relative differences for all six recorded quantities and applies configurable mode-specific tolerances.

In [ ]:
NUMERICAL_COLUMNS = [
    "run_label",
    "orientation_key",
    "orientation_index",
    "azimuth_degrees",
    "polar_degrees",
    "comparison",
    "mode",
    "fingerprint",
    "reference_value",
    "comparison_value",
    "difference",
    "absolute_error",
    "relative_error",
    "tolerance",
    "within_tolerance",
]


def numerical_difference(
    row: pd.Series,
    *,
    comparison: str,
    mode: str,
    fingerprint: str,
    reference_value: float,
    comparison_value: float,
    rtol: float,
    atol: float,
) -> dict[str, Any] | None:
    if not np.isfinite(reference_value) or not np.isfinite(comparison_value):
        return None
    difference = float(comparison_value - reference_value)
    absolute_error = abs(difference)
    relative_error = absolute_error / max(
        abs(float(reference_value)), np.finfo(float).tiny
    )
    tolerance = atol + rtol * abs(float(reference_value))
    return {
        "run_label": row["run_label"],
        "orientation_key": row["orientation_key"],
        "orientation_index": row["orientation_index"],
        "azimuth_degrees": row["azimuth_degrees"],
        "polar_degrees": row["polar_degrees"],
        "comparison": comparison,
        "mode": mode,
        "fingerprint": fingerprint,
        "reference_value": float(reference_value),
        "comparison_value": float(comparison_value),
        "difference": difference,
        "absolute_error": absolute_error,
        "relative_error": relative_error,
        "tolerance": tolerance,
        "within_tolerance": absolute_error <= tolerance,
    }


numerical_rows: list[dict[str, Any]] = []
for _, row in normalized_df.iterrows():
    mode = str(row["mode"])
    rtol, atol = ERROR_TOLERANCES.get(mode, (5e-8, 1e-8))
    for fingerprint in FINGERPRINT_LABELS:
        record = numerical_difference(
            row,
            comparison="runtime_vs_memory",
            mode=mode,
            fingerprint=fingerprint,
            reference_value=float(row[f"runtime_{fingerprint}"]),
            comparison_value=float(row[f"memory_{fingerprint}"]),
            rtol=rtol,
            atol=atol,
        )
        if record is not None:
            numerical_rows.append(record)

if not normalized_df.empty:
    indexed = normalized_df.set_index(
        ["run_label", "orientation_key", "mode"], drop=False
    )
    for (run_label, orientation_key), _ in normalized_df.groupby(
        ["run_label", "orientation_key"]
    ):
        dense_key = (run_label, orientation_key, REFERENCE_MODE)
        if dense_key not in indexed.index:
            continue
        dense_row = indexed.loc[dense_key]
        for mode in ("cpu_screened", "gpu"):
            production_key = (run_label, orientation_key, mode)
            if production_key not in indexed.index:
                continue
            production_row = indexed.loc[production_key]
            rtol, atol = ERROR_TOLERANCES[mode]
            for fingerprint in FINGERPRINT_LABELS:
                record = numerical_difference(
                    production_row,
                    comparison="mode_vs_cpu_dense",
                    mode=mode,
                    fingerprint=fingerprint,
                    reference_value=float(dense_row[f"runtime_{fingerprint}"]),
                    comparison_value=float(production_row[f"runtime_{fingerprint}"]),
                    rtol=rtol,
                    atol=atol,
                )
                if record is not None:
                    numerical_rows.append(record)

numerical_df = pd.DataFrame(numerical_rows, columns=NUMERICAL_COLUMNS)
if numerical_df.empty:
    numerical_summary_df = pd.DataFrame()
else:
    numerical_summary_df = (
        numerical_df.groupby(["run_label", "comparison", "mode", "fingerprint"])
        .agg(
            compared=("absolute_error", "size"),
            max_absolute_error=("absolute_error", "max"),
            max_relative_error=("relative_error", "max"),
            outside_tolerance=("within_tolerance", lambda values: int((~values).sum())),
        )
        .reset_index()
    )
display(numerical_summary_df)

## 7. Calculate Runtime Metrics and Speedups

Runtime summaries use the median sample as the representative value. The comparison table includes dense-to-screened, dense-to-GPU, screened-CPU-to-GPU, and cross-run speedups at matched orientations.

## 8. Calculate Memory Metrics and Reductions

Incremental peaks and GPU allocator baselines are converted to GiB. Reduction factors use CPU dense as the numerator so values above one indicate improvement.

## 9. Analyze Scaling with Molecular Size

This benchmark intentionally fixes molecular size at C7H16 and 879 AOs, so molecular-size fitting is not meaningful. Instead, a harmonic least-squares model summarizes orientation sensitivity and records fit coefficients, $R^2$, and residual RMSE for runtime and memory.

In [ ]:
def summarize_metric(frame: pd.DataFrame, metric: str, value_name: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for (run_label, mode), group in frame.groupby(["run_label", "mode"]):
        values = group[metric].dropna().astype(float).to_numpy()
        if values.size == 0:
            continue
        mean_value = float(values.mean())
        rows.append(
            {
                "run_label": run_label,
                "mode": mode,
                "observation_count": int(values.size),
                "minimum": float(values.min()),
                "median": float(np.median(values)),
                "mean": mean_value,
                "standard_deviation": (
                    float(values.std(ddof=1)) if values.size > 1 else 0.0
                ),
                "coefficient_of_variation": (
                    float(values.std(ddof=1) / mean_value)
                    if values.size > 1 and mean_value != 0.0
                    else 0.0
                ),
                "representative": float(np.median(values)),
                "unit": value_name,
            }
        )
    return pd.DataFrame(rows)


runtime_statistics_df = summarize_metric(normalized_df, "runtime_seconds", "seconds")
if not runtime_statistics_df.empty:
    runtime_sample_counts = (
        normalized_df.assign(
            runtime_sample_count=normalized_df["runtime_samples_seconds"].map(len)
        )
        .groupby(["run_label", "mode"])["runtime_sample_count"]
        .sum()
        .reset_index()
    )
    runtime_statistics_df = runtime_statistics_df.merge(
        runtime_sample_counts,
        on=["run_label", "mode"],
        how="left",
    )
memory_statistics_df = summarize_metric(normalized_df, "incremental_peak_gib", "GiB")


def finite_ratio(numerator: Any, denominator: Any) -> float:
    numerator_value = float(numerator)
    denominator_value = float(denominator)
    if (
        np.isfinite(numerator_value)
        and np.isfinite(denominator_value)
        and denominator_value > 0.0
    ):
        return numerator_value / denominator_value
    return np.nan


comparison_rows: list[dict[str, Any]] = []
for (run_label, orientation_key), group in normalized_df.groupby(
    ["run_label", "orientation_key"]
):
    by_mode = group.set_index("mode")
    if any(mode not in by_mode.index for mode in MODES):
        continue
    dense = by_mode.loc["cpu_dense"]
    screened = by_mode.loc["cpu_screened"]
    gpu = by_mode.loc["gpu"]
    comparison_rows.append(
        {
            "run_label": run_label,
            "orientation_key": orientation_key,
            "orientation_index": dense["orientation_index"],
            "azimuth_degrees": dense["azimuth_degrees"],
            "polar_degrees": dense["polar_degrees"],
            "dense_to_screened_runtime_speedup": finite_ratio(
                dense["runtime_seconds"], screened["runtime_seconds"]
            ),
            "dense_to_gpu_runtime_speedup": finite_ratio(
                dense["runtime_seconds"], gpu["runtime_seconds"]
            ),
            "screened_cpu_to_gpu_runtime_speedup": finite_ratio(
                screened["runtime_seconds"], gpu["runtime_seconds"]
            ),
            "dense_to_screened_memory_reduction": finite_ratio(
                dense["incremental_peak_gib"], screened["incremental_peak_gib"]
            ),
            "dense_to_gpu_memory_reduction": finite_ratio(
                dense["incremental_peak_gib"], gpu["incremental_peak_gib"]
            ),
            "screened_cpu_to_gpu_memory_ratio": finite_ratio(
                screened["incremental_peak_gib"], gpu["incremental_peak_gib"]
            ),
        }
    )
comparison_metrics_df = pd.DataFrame(comparison_rows)

cross_run_rows: list[dict[str, Any]] = []
if len(DOCUMENTS) > 1 and not normalized_df.empty:
    baseline_path, baseline_document = DOCUMENTS[0]
    baseline_run_label = str(baseline_document.get("run_label") or baseline_path.stem)
    baseline = normalized_df[
        normalized_df["run_label"] == baseline_run_label
    ].set_index(["orientation_key", "mode"])
    for current_path, document in DOCUMENTS[1:]:
        current_run_label = str(document.get("run_label") or current_path.stem)
        current = normalized_df[
            normalized_df["run_label"] == current_run_label
        ].set_index(["orientation_key", "mode"])
        for key in baseline.index.intersection(current.index):
            baseline_row = baseline.loc[key]
            current_row = current.loc[key]
            cross_run_rows.append(
                {
                    "baseline_run_label": baseline_run_label,
                    "comparison_run_label": current_run_label,
                    "orientation_key": key[0],
                    "mode": key[1],
                    "runtime_speedup": finite_ratio(
                        baseline_row["runtime_seconds"],
                        current_row["runtime_seconds"],
                    ),
                    "memory_reduction": finite_ratio(
                        baseline_row["incremental_peak_gib"],
                        current_row["incremental_peak_gib"],
                    ),
                }
            )
cross_run_df = pd.DataFrame(cross_run_rows)

orientation_fit_rows: list[dict[str, Any]] = []
for (run_label, mode), group in normalized_df.groupby(["run_label", "mode"]):
    for metric in ("runtime_seconds", "incremental_peak_gib"):
        fit_data = group.dropna(subset=["azimuth_degrees", "polar_degrees", metric])
        if len(fit_data) < 5:
            continue
        azimuth = np.radians(fit_data["azimuth_degrees"].to_numpy(float))
        polar = np.radians(fit_data["polar_degrees"].to_numpy(float))
        design = np.column_stack(
            [
                np.ones(len(fit_data)),
                np.sin(azimuth),
                np.cos(azimuth),
                np.sin(polar),
                np.cos(polar),
            ]
        )
        values = fit_data[metric].to_numpy(float)
        coefficients, _, _, _ = np.linalg.lstsq(design, values, rcond=None)
        residuals = values - design @ coefficients
        total_variation = float(np.square(values - values.mean()).sum())
        residual_variation = float(np.square(residuals).sum())
        orientation_fit_rows.append(
            {
                "run_label": run_label,
                "mode": mode,
                "metric": metric,
                "intercept": float(coefficients[0]),
                "sin_azimuth": float(coefficients[1]),
                "cos_azimuth": float(coefficients[2]),
                "sin_polar": float(coefficients[3]),
                "cos_polar": float(coefficients[4]),
                "r_squared": (
                    1.0 - residual_variation / total_variation
                    if total_variation > 0.0
                    else 1.0
                ),
                "residual_rmse": float(np.sqrt(np.mean(np.square(residuals)))),
            }
        )
orientation_fit_df = pd.DataFrame(orientation_fit_rows)

print("Runtime statistics")
display(runtime_statistics_df)
print("Memory statistics")
display(memory_statistics_df)
print("Per-orientation speedup and reduction metrics")
display(comparison_metrics_df)
print("Cross-run changes")
display(cross_run_df)
print("Orientation-sensitivity fits")
display(orientation_fit_df)

## 10. Visualize Runtime Comparisons

Runtime is shown by orientation index, as mode/revision distributions, and as 12-by-6 azimuth/polar heatmaps. Since AO count is fixed, route labels replace a screening-threshold marker.

## 11. Visualize Memory Comparisons

Incremental peak memory uses the same views, with GPU allocator baseline retained in the normalized table and exported metadata.

## 12. Visualize Speedup and Memory Reduction

Dense-to-screened and dense-to-GPU factors are rendered on the same angular grid. Values above one indicate faster execution or lower peak memory than CPU dense.

## 13. Visualize Numerical Differences

Signed fingerprint differences use CPU dense as zero. Change `FINGERPRINT_TO_PLOT` to inspect any of the six recorded fingerprint quantities.

In [ ]:
import re

from matplotlib.figure import Figure

LINE_STYLES = ("-", "--", "-.", ":")
MARKERS = ("o", "s", "^", "D")
PLOT_STYLES = tuple(zip(LINE_STYLES, MARKERS, strict=True))
if len(DOCUMENTS) > len(PLOT_STYLES):
    raise ValueError(f"At most {len(PLOT_STYLES)} distinct run labels can be plotted")
LABEL_PLOT_STYLES = {
    str(document.get("run_label") or path.stem): PLOT_STYLES[index]
    for index, (path, document) in enumerate(DOCUMENTS)
}


def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", value).strip("-.") or "result"


def orientation_matrix(frame: pd.DataFrame, value_column: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    return (
        frame.pivot(
            index="polar_degrees",
            columns="azimuth_degrees",
            values=value_column,
        )
        .sort_index()
        .sort_index(axis=1)
    )


def finite_value_range(values: Any) -> tuple[float, float] | None:
    numeric = np.asarray(values, dtype=float)
    finite = numeric[np.isfinite(numeric)]
    if finite.size == 0:
        return None
    minimum = float(finite.min())
    maximum = float(finite.max())
    if minimum == maximum:
        padding = max(abs(minimum) * 1e-9, np.finfo(float).eps)
        return minimum - padding, maximum + padding
    return minimum, maximum


def observed_range_errors(samples: Any, center: float) -> tuple[float, float]:
    numeric = np.asarray(samples, dtype=float)
    finite = numeric[np.isfinite(numeric)]
    if finite.size == 0:
        return 0.0, 0.0
    return (
        max(0.0, center - float(finite.min())),
        max(0.0, float(finite.max()) - center),
    )


def finish_figure(figure: Figure, output_path: Path | None = None) -> None:
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()


def plot_orientation_lines(
    value_column: str,
    ylabel: str,
    output_path: Path | None = None,
    error_samples_column: str | None = None,
) -> None:
    data = normalized_df.dropna(subset=["orientation_index", value_column])
    if data.empty:
        print(f"No successful values available for {value_column}")
        return
    figure, axis = plt.subplots(figsize=(12, 6), constrained_layout=True)
    for (run_label, mode), group in data.groupby(["run_label", "mode"]):
        ordered = group.sort_values("orientation_index")
        x_values = ordered["orientation_index"].to_numpy(float)
        centers = ordered[value_column].to_numpy(float)
        line_style, marker = LABEL_PLOT_STYLES[str(run_label)]
        plot_options = {
            "color": MODE_COLORS[mode],
            "linewidth": 1.2,
            "alpha": 0.85,
            "label": f"{run_label} {mode}",
            "linestyle": line_style,
            "marker": marker,
            "markersize": 3.5,
            "markevery": max(1, len(ordered) // 12),
        }
        if error_samples_column is None:
            axis.plot(x_values, centers, **plot_options)
        else:
            errors = np.asarray(
                [
                    observed_range_errors(samples, center)
                    for samples, center in zip(
                        ordered[error_samples_column], centers, strict=True
                    )
                ],
                dtype=float,
            ).T
            axis.errorbar(
                x_values,
                centers,
                yerr=errors,
                capsize=2,
                elinewidth=0.7,
                **plot_options,
            )
    title = f"{ylabel} across molecular orientations"
    if error_samples_column is not None:
        title += " (median and observed min-max)"
    axis.set(
        title=title,
        xlabel="Orientation index (azimuth-major, then polar)",
        ylabel=ylabel,
    )
    axis.grid(True, color="#D9D9D9", linewidth=0.6)
    axis.legend(fontsize=8, ncol=2)
    finish_figure(figure, output_path)


def plot_mode_distribution(
    value_column: str, ylabel: str, output_path: Path | None = None
) -> None:
    data = normalized_df.dropna(subset=[value_column])
    if data.empty:
        print(f"No successful values available for {value_column}")
        return
    figure, axis = plt.subplots(figsize=(10, 6), constrained_layout=True)
    sns.boxplot(
        data=data,
        x="mode",
        y=value_column,
        hue="run_label",
        order=MODES,
        showfliers=True,
        ax=axis,
    )
    axis.set(title=f"{ylabel} distribution by mode", xlabel="Mode", ylabel=ylabel)
    axis.grid(True, axis="y", color="#D9D9D9", linewidth=0.6)
    finish_figure(figure, output_path)


def plot_measurement_heatmaps(
    value_column: str,
    colorbar_label: str,
    output_dir: Path | None = None,
) -> None:
    if normalized_df[value_column].dropna().empty:
        print(f"No successful values available for {value_column}")
        return
    for path, document in DOCUMENTS:
        run_label = str(document.get("run_label") or path.stem)
        run_data = normalized_df[normalized_df["run_label"] == run_label]
        figure, axes = plt.subplots(
            1, len(MODES), figsize=(18, 4.8), constrained_layout=True
        )
        for axis, mode in zip(axes, MODES, strict=True):
            matrix = orientation_matrix(
                run_data[run_data["mode"] == mode], value_column
            )
            value_range = finite_value_range(matrix)
            if value_range is None:
                axis.text(0.5, 0.5, "No successful data", ha="center", va="center")
                axis.set_axis_off()
                continue
            minimum, maximum = value_range
            sns.heatmap(
                matrix,
                mask=matrix.isna(),
                cmap="viridis",
                vmin=minimum,
                vmax=maximum,
                cbar_kws={"label": colorbar_label},
                ax=axis,
            )
            route_values = (
                run_data.loc[run_data["mode"] == mode, "selected_route"]
                .dropna()
                .unique()
            )
            route_label = ", ".join(str(value) for value in route_values) or "no route"
            axis.set(
                title=f"{mode} ({route_label})",
                xlabel="Azimuth (degrees)",
                ylabel="Polar angle (degrees)",
            )
        figure.suptitle(f"{run_label} {colorbar_label} by orientation")
        output_path = (
            output_dir
            / f"{safe_filename(run_label)}-{safe_filename(value_column)}-heatmap.png"
            if output_dir is not None
            else None
        )
        finish_figure(figure, output_path)

In [ ]:
RATIO_LABELS = {
    "dense_to_screened_runtime_speedup": "CPU dense / CPU screened runtime",
    "dense_to_gpu_runtime_speedup": "CPU dense / GPU runtime",
    "dense_to_screened_memory_reduction": "CPU dense / CPU screened peak",
    "dense_to_gpu_memory_reduction": "CPU dense / GPU peak",
}


def plot_ratio_heatmaps(
    ratio_columns: tuple[str, ...],
    title: str,
    output_dir: Path | None = None,
) -> None:
    if comparison_metrics_df.empty:
        print(f"No matched mode data available for {title}")
        return
    for path, document in DOCUMENTS:
        run_label = str(document.get("run_label") or path.stem)
        run_data = comparison_metrics_df[
            comparison_metrics_df["run_label"] == run_label
        ]
        figure, axes = plt.subplots(
            1, len(ratio_columns), figsize=(12, 4.8), constrained_layout=True
        )
        axes_array = np.atleast_1d(axes)
        for axis, ratio_column in zip(axes_array, ratio_columns, strict=True):
            matrix = orientation_matrix(run_data, ratio_column)
            value_range = finite_value_range(matrix)
            if value_range is None:
                axis.text(0.5, 0.5, "No successful data", ha="center", va="center")
                axis.set_axis_off()
                continue
            minimum, maximum = value_range
            heatmap_options: dict[str, Any] = {
                "cmap": "RdYlGn",
                "vmin": minimum,
                "vmax": maximum,
            }
            if minimum < 1.0 < maximum:
                heatmap_options["center"] = 1.0
            sns.heatmap(
                matrix,
                mask=matrix.isna(),
                cbar_kws={"label": "Factor"},
                ax=axis,
                **heatmap_options,
            )
            axis.set(
                title=RATIO_LABELS[ratio_column],
                xlabel="Azimuth (degrees)",
                ylabel="Polar angle (degrees)",
            )
        figure.suptitle(f"{run_label} {title}")
        output_path = (
            output_dir / f"{safe_filename(run_label)}-{safe_filename(title)}.png"
            if output_dir is not None
            else None
        )
        finish_figure(figure, output_path)


def plot_fingerprint_heatmaps(fingerprint: str, output_dir: Path | None = None) -> None:
    if fingerprint not in FINGERPRINT_LABELS:
        raise ValueError(f"Unknown fingerprint {fingerprint!r}")
    data = numerical_df[
        (numerical_df["comparison"] == "mode_vs_cpu_dense")
        & (numerical_df["fingerprint"] == fingerprint)
    ]
    if data.empty:
        print(f"No matched numerical data available for {fingerprint}")
        return
    for run_label, run_data in data.groupby("run_label"):
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
        for axis, mode in zip(axes, ("cpu_screened", "gpu"), strict=True):
            matrix = orientation_matrix(
                run_data[run_data["mode"] == mode], "difference"
            )
            value_range = finite_value_range(matrix)
            if value_range is None:
                axis.text(0.5, 0.5, "No successful data", ha="center", va="center")
                axis.set_axis_off()
                continue
            minimum, maximum = value_range
            heatmap_options = {
                "cmap": "coolwarm",
                "vmin": minimum,
                "vmax": maximum,
            }
            if minimum < 0.0 < maximum:
                heatmap_options["center"] = 0.0
            sns.heatmap(
                matrix,
                mask=matrix.isna(),
                cbar_kws={"label": "Mode - CPU dense"},
                ax=axis,
                **heatmap_options,
            )
            axis.set(
                title=mode,
                xlabel="Azimuth (degrees)",
                ylabel="Polar angle (degrees)",
            )
        figure.suptitle(f"{run_label} {FINGERPRINT_LABELS[fingerprint]} difference")
        output_path = (
            output_dir
            / f"{safe_filename(str(run_label))}-{safe_filename(fingerprint)}-difference.png"
            if output_dir is not None
            else None
        )
        finish_figure(figure, output_path)

In [ ]:
FINGERPRINT_TO_PLOT = "xc_energy"

plot_orientation_lines(
    "runtime_seconds",
    "Runtime (s)",
    error_samples_column="runtime_samples_seconds",
)
plot_mode_distribution("runtime_seconds", "Runtime (s)")
plot_measurement_heatmaps("runtime_seconds", "Runtime (s)")

plot_orientation_lines("incremental_peak_gib", "Incremental peak memory (GiB)")
plot_mode_distribution("incremental_peak_gib", "Incremental peak memory (GiB)")
plot_measurement_heatmaps("incremental_peak_gib", "Incremental peak memory (GiB)")

plot_ratio_heatmaps(
    ("dense_to_screened_runtime_speedup", "dense_to_gpu_runtime_speedup"),
    "runtime speedup",
)
plot_ratio_heatmaps(
    ("dense_to_screened_memory_reduction", "dense_to_gpu_memory_reduction"),
    "memory reduction",
)
plot_fingerprint_heatmaps(FINGERPRINT_TO_PLOT)

## 14. Record Environment and Source Metadata

This table keeps hardware, package versions, CUDA details, thread settings, scientific configuration, source commit, dirty state, implementation hash, and both runner hashes alongside every comparison.

## 15. Export Comparison Results to JSON

The comparison artifact contains JSON-safe configuration summaries, normalized rows, runtime and memory statistics, speedups, numerical differences, validation failures, angular fits, environment metadata, and source provenance.

## 16. Save Tables and Figures

The final cell writes CSV tables and deterministic PNG files below `benchmarks/results/rotation_comparison`. Re-running the cell refreshes the report artifacts from the currently selected input files.

In [ ]:
metadata_rows: list[dict[str, Any]] = []
validation_failure_counts = Counter(row["run_label"] for row in validation_rows)
for path, document in DOCUMENTS:
    environment = document.get("environment", {})
    packages = environment.get("packages", {})
    cuda = environment.get("cuda", {})
    configuration = document.get("configuration", {})
    source = document.get("source", {})
    hashes = document.get("runner_hashes", {})
    run_label = str(document.get("run_label") or path.stem)
    document_rows = normalized_df[normalized_df["run_label"] == run_label]
    implementation_hashes = sorted(
        str(value) for value in document_rows["implementation_sha256"].dropna().unique()
    )
    metadata_rows.append(
        {
            "run_label": run_label,
            "created_at": document.get("created_at"),
            "updated_at": document.get("updated_at"),
            "python": environment.get("python"),
            "python_executable": environment.get("python_executable"),
            "pyscf": packages.get("pyscf"),
            "skala": packages.get("skala"),
            "torch": packages.get("torch"),
            "cupy": packages.get("cupy"),
            "gpu4pyscf": packages.get("gpu4pyscf"),
            "memray": packages.get("memray"),
            "cuda_available": cuda.get("available"),
            "torch_cuda_version": cuda.get("torch_cuda_version"),
            "device_name": cuda.get("device_name"),
            "cpu_threads": configuration.get("cpu_threads"),
            "thread_environment": environment.get("thread_environment"),
            "basis": configuration.get("basis"),
            "functional": configuration.get("functional"),
            "grid_level": configuration.get("grid_level"),
            "grid_alignment": configuration.get("grid_alignment"),
            "max_memory_mb": configuration.get("max_memory_mb"),
            "orientation_count": configuration.get("orientation_count"),
            "commit": source.get("commit"),
            "branch": source.get("branch"),
            "dirty": source.get("dirty"),
            "implementation_hashes": implementation_hashes,
            "worker_sha256": hashes.get("worker_sha256"),
            "rotation_runner_sha256": hashes.get("rotation_runner_sha256"),
            "validation_failure_count": validation_failure_counts[run_label],
        }
    )
metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)

In [ ]:
def records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    return frame.to_dict(orient="records") if not frame.empty else []


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if value is pd.NA:
        return None
    return value


comparison_document = {
    "schema_version": 1,
    "benchmark": "pyscf_ao_screening_rotation_comparison",
    "generated_at": datetime.now(UTC).isoformat(),
    "selected_run_labels": [
        str(document.get("run_label") or path.stem) for path, document in DOCUMENTS
    ],
    "configuration_summaries": [
        {
            "run_label": str(document.get("run_label") or path.stem),
            "configuration": document.get("configuration"),
        }
        for path, document in DOCUMENTS
    ],
    "normalized_measurements": records(normalized_df),
    "runtime_statistics": records(runtime_statistics_df),
    "memory_statistics": records(memory_statistics_df),
    "speedups_and_memory_reductions": records(comparison_metrics_df),
    "cross_run_changes": records(cross_run_df),
    "numerical_differences": records(numerical_df),
    "numerical_summary": records(numerical_summary_df),
    "orientation_sensitivity_fits": records(orientation_fit_df),
    "validation_failures": records(validation_df),
    "environment_metadata": records(metadata_df),
    "source_provenance": [
        {
            "run_label": str(document.get("run_label") or path.stem),
            "source": document.get("source"),
            "runner_hashes": document.get("runner_hashes"),
        }
        for path, document in DOCUMENTS
    ],
}
comparison_document = json_safe(comparison_document)
print(
    f"Prepared comparison JSON with "
    f"{len(comparison_document['normalized_measurements'])} normalized rows"
)

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

with COMPARISON_JSON.open("w", encoding="utf-8") as stream:
    json.dump(comparison_document, stream, indent=2, sort_keys=True, allow_nan=False)
    stream.write("\n")

tables = {
    "normalized-measurements.csv": normalized_df,
    "validation-failures.csv": validation_df,
    "status-summary.csv": status_summary_df,
    "route-summary.csv": route_summary_df,
    "runtime-statistics.csv": runtime_statistics_df,
    "memory-statistics.csv": memory_statistics_df,
    "speedups-and-memory-reductions.csv": comparison_metrics_df,
    "cross-run-changes.csv": cross_run_df,
    "numerical-differences.csv": numerical_df,
    "numerical-summary.csv": numerical_summary_df,
    "orientation-sensitivity-fits.csv": orientation_fit_df,
    "environment-and-source-metadata.csv": metadata_df,
}
for filename, table in tables.items():
    table.to_csv(TABLE_DIR / filename, index=False)

plot_orientation_lines(
    "runtime_seconds",
    "Runtime (s)",
    FIGURE_DIR / "runtime-by-orientation.png",
)
plot_mode_distribution(
    "runtime_seconds",
    "Runtime (s)",
    FIGURE_DIR / "runtime-by-mode.png",
)
plot_measurement_heatmaps("runtime_seconds", "Runtime (s)", FIGURE_DIR)
plot_orientation_lines(
    "incremental_peak_gib",
    "Incremental peak memory (GiB)",
    FIGURE_DIR / "memory-by-orientation.png",
)
plot_mode_distribution(
    "incremental_peak_gib",
    "Incremental peak memory (GiB)",
    FIGURE_DIR / "memory-by-mode.png",
)
plot_measurement_heatmaps(
    "incremental_peak_gib", "Incremental peak memory (GiB)", FIGURE_DIR
)
plot_ratio_heatmaps(
    ("dense_to_screened_runtime_speedup", "dense_to_gpu_runtime_speedup"),
    "runtime speedup",
    FIGURE_DIR,
)
plot_ratio_heatmaps(
    ("dense_to_screened_memory_reduction", "dense_to_gpu_memory_reduction"),
    "memory reduction",
    FIGURE_DIR,
)
for fingerprint in FINGERPRINT_LABELS:
    plot_fingerprint_heatmaps(fingerprint, FIGURE_DIR)

print(f"Comparison JSON: {COMPARISON_JSON}")
print(f"Tables: {TABLE_DIR}")
print(f"Figures: {FIGURE_DIR}")